In [ ]:
#import statements
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Data Overview


In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("/content/drive/MyDrive/combined_data.csv", low_memory=False, na_values=['', ' '])
df.shape

In [ ]:
df.head()

,INCIDENT_NUMBER,OFFENSE_CODE,OFFENSE_CODE_GROUP,OFFENSE_DESCRIPTION,DISTRICT,REPORTING_AREA,SHOOTING,OCCURRED_ON_DATE,YEAR,MONTH,DAY_OF_WEEK,HOUR,UCR_PART,STREET,Lat,Long,Location
0,212012996,1102,NaN,FRAUD - FALSE PRETENSE / SCHEME,B2,259,0,2020-01-01 00:00:00,2020,1,Wednesday,0,NaN,ALEXANDER ST,42.316942,-71.069912,"(42.3169424347871, -71.0699121273113)"
1,212008096,1107,NaN,FRAUD - IMPERSONATION,A1,118,0,2020-01-01 00:00:00,2020,1,Wednesday,0,NaN,BOYLSTON ST,42.352418,-71.065255,"(42.3524181472861, -71.0652549858121)"
2,202000034,3006,NaN,SICK/INJURED/MEDICAL - PERSON,C6,201,0,2020-01-01 00:00:00,2020,1,Wednesday,0,NaN,W BROADWAY,42.340070,-71.052794,"(42.340069862647, -71.0527942008028)"
3,202007210,1001,NaN,FORGERY / COUNTERFEITING,C6,200,0,2020-01-01 00:00:00,2020,1,Wednesday,0,NaN,ORTON-MAROTTA WAY,42.341288,-71.054679,"(42.3412875043904, -71.054679326494)"
4,202000355,617,NaN,LARCENY THEFT FROM BUILDING,A1,77,0,2020-01-01 00:00:00,2020,1,Wednesday,0,NaN,FRIEND ST,42.361839,-71.059765,"(42.3618385665647, -71.0597648909416)"


In [ ]:
df.describe()

,OFFENSE_CODE,OFFENSE_CODE_GROUP,SHOOTING,YEAR,MONTH,HOUR,UCR_PART,Lat,Long
count,433825.000000,0.0,433825.000000,433825.000000,433825.000000,433825.000000,0.0,4.112140e+05,4.112140e+05
mean,2350.568195,NaN,0.009935,2022.469062,6.426289,12.644912,NaN,4.232296e+01,-7.108351e+01
std,1192.202433,NaN,0.099178,1.660890,3.328813,6.481940,NaN,7.345040e-02,1.152090e-01
min,111.000000,NaN,0.000000,2020.000000,1.000000,0.000000,NaN,1.327276e-07,-7.134947e+01
25%,1102.000000,NaN,0.000000,2021.000000,4.000000,9.000000,NaN,4.229755e+01,-7.109891e+01
50%,3005.000000,NaN,0.000000,2023.000000,6.000000,13.000000,NaN,4.232866e+01,-7.107754e+01
75%,3201.000000,NaN,0.000000,2024.000000,9.000000,18.000000,NaN,4.234893e+01,-7.106096e+01
max,99999.000000,NaN,1.000000,2025.000000,12.000000,23.000000,NaN,4.246141e+01,5.249691e-08


In [ ]:
df.dtypes

,0
INCIDENT_NUMBER,object
OFFENSE_CODE,int64
OFFENSE_CODE_GROUP,float64
OFFENSE_DESCRIPTION,object
DISTRICT,object
REPORTING_AREA,object
SHOOTING,int64
OCCURRED_ON_DATE,object
YEAR,int64
MONTH,int64


In [ ]:
missing_counts = df.isna().sum()

print("Missing values per column:")
print(missing_counts)

Missing values per column:
INCIDENT_NUMBER             0
OFFENSE_CODE                0
OFFENSE_CODE_GROUP     433825
OFFENSE_DESCRIPTION         0
DISTRICT                 1879
REPORTING_AREA         100787
SHOOTING                    0
OCCURRED_ON_DATE            0
YEAR                        0
MONTH                       0
DAY_OF_WEEK                 0
HOUR                        0
UCR_PART               433825
STREET                    681
Lat                     22611
Long                    22611
Location                22611
dtype: int64


In [ ]:
df_month = df.dropna(subset=['YEAR', 'MONTH'])

# Ensure 'OCCURRED_ON_DATE' is in datetime format and set to the start of the month
df_month['OCCURRED_ON_DATE'] = pd.to_datetime(df_month['OCCURRED_ON_DATE'], errors='coerce').dt.to_period('M').dt.to_timestamp()

# Group by Year and Month and count incidents
monthly = df_month.groupby(['YEAR', 'MONTH']).size().reset_index(name='Incidents')

# Create a YearMonth column for plotting
monthly['YearMonth'] = pd.to_datetime(monthly['YEAR'].astype(str) + '-' + monthly['MONTH'].astype(str) + '-01')

# Ensure the data only goes up to September 2025 (as per previous analysis)
monthly = monthly[monthly['YearMonth'] <= "2025-09-01"]

# --- Plot ---
plt.figure(figsize=(14,6))
plt.plot(monthly['YearMonth'], monthly['Incidents'], marker='o', color="firebrick")

plt.title("Monthly Crime Incidents in Boston (2020–Sep 2025) With Uncleaned Dataset", fontsize=20)
plt.xlabel("Month", fontsize=15)  # Increased font size
plt.ylabel("Number of Incidents", fontsize=20) # Increased font size
plt.grid(alpha=0.3)

# Format x-axis to show months
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))  # e.g., Jan 2024

plt.xticks(rotation=45, ha='right', fontsize=15) # Increased font size for ticks
plt.yticks(fontsize=20) # Increased font size for ticks
plt.tight_layout()
plt.savefig("monthly_incidents.eps", format="eps", dpi=300, bbox_inches="tight")
plt.show()


# Data Preprocessing and Cleaning

In [ ]:
# Drop OFFENSE_CODE_GROUP and UCR_PART columns (empty columns)
cols_to_drop = ["OFFENSE_CODE_GROUP", "UCR_PART"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

# Drop rows with missing location information
df = df.dropna(subset=["Location", "DISTRICT"])

# df is now your cleaned dataset
print(f"Cleaned dataset loaded into df, shape = {df.shape}")

Cleaned dataset loaded into df, shape = (410002, 15)


In [ ]:
counts_by_year = df['YEAR'].value_counts().sort_index()
total_count = counts_by_year.sum()

print(counts_by_year)
print("\nTotal incidents:", total_count)

YEAR
2020    68928
2021    67983
2022    70008
2023    72588
2024    74625
2025    55870
Name: count, dtype: int64

Total incidents: 410002


In [ ]:
# Count the number of incidents for each offense description
offense_counts = df['OFFENSE_DESCRIPTION'].value_counts()

# Display the counts
print("Number of incidents per offense description:")
print(offense_counts.head(20))

Number of incidents per offense description:
OFFENSE_DESCRIPTION
INVESTIGATE PERSON                              39418
SICK ASSIST                                     32096
M/V - LEAVING SCENE - PROPERTY DAMAGE           24038
INVESTIGATE PROPERTY                            18996
TOWED MOTOR VEHICLE                             17634
VANDALISM                                       16399
ASSAULT - SIMPLE                                16280
LARCENY SHOPLIFTING                             15774
PROPERTY - LOST/ MISSING                        12193
LARCENY THEFT FROM MV - NON-ACCESSORY           11163
M/V ACCIDENT - PROPERTY DAMAGE                  10285
VERBAL DISPUTE                                   9889
DRUGS - POSSESSION/ SALE/ MANUFACTURING/ USE     9738
LARCENY THEFT FROM BUILDING                      9560
THREATS TO DO BODILY HARM                        9468
ASSAULT - AGGRAVATED                             8859
SICK/INJURED/MEDICAL - PERSON                    8553
LARCENY ALL OTHER

In [ ]:
# List of offense descriptions to drop (unlikely to spillover)
drop_offenses = [
    "SICK ASSIST",
    "SICK/INJURED/MEDICAL - PERSON",
    "TOWED MOTOR VEHICLE",
    "INVESTIGATE PERSON",
    "INVESTIGATE PROPERTY"
]

# drop from df
df = df[~df["OFFENSE_DESCRIPTION"].isin(drop_offenses)].copy()

offense_counts = df['OFFENSE_DESCRIPTION'].value_counts()
print(offense_counts)
print(df.shape)

OFFENSE_DESCRIPTION
M/V - LEAVING SCENE - PROPERTY DAMAGE    24038
VANDALISM                                16399
ASSAULT - SIMPLE                         16280
LARCENY SHOPLIFTING                      15774
PROPERTY - LOST/ MISSING                 12193
                                         ...  
Justifiable Homicide                         2
PROSTITUTION - ASSISTING OR PROMOTING        2
Evidence Tracker Incidents                   1
MANSLAUGHTER - NEGLIGENCE                    1
PRISONER ESCAPE / ESCAPE & RECAPTURE         1
Name: count, Length: 122, dtype: int64
(293305, 15)


In [ ]:
missing_counts = df.isna().sum()

print("Missing values per column:")
print(missing_counts)


Missing values per column:
INCIDENT_NUMBER            0
OFFENSE_CODE               0
OFFENSE_DESCRIPTION        0
DISTRICT                   0
REPORTING_AREA         66536
SHOOTING                   0
OCCURRED_ON_DATE           0
YEAR                       0
MONTH                      0
DAY_OF_WEEK                0
HOUR                       0
STREET                   516
Lat                        0
Long                       0
Location                   0
dtype: int64


In [ ]:
df_month = df.dropna(subset=['YEAR', 'MONTH'])

# Ensure 'OCCURRED_ON_DATE' is in datetime format and set to the start of the month
df_month['OCCURRED_ON_DATE'] = pd.to_datetime(df_month['OCCURRED_ON_DATE'], errors='coerce').dt.to_period('M').dt.to_timestamp()

# Group by Year and Month and count incidents
monthly = df_month.groupby(['YEAR', 'MONTH']).size().reset_index(name='Incidents')

# Create a YearMonth column for plotting
monthly['YearMonth'] = pd.to_datetime(monthly['YEAR'].astype(str) + '-' + monthly['MONTH'].astype(str) + '-01')

# Ensure the data only goes up to September 2025 (as per previous analysis)
monthly = monthly[monthly['YearMonth'] <= "2025-09-01"]

# --- Plot ---
plt.figure(figsize=(14,6))
plt.plot(monthly['YearMonth'], monthly['Incidents'], marker='o', color="firebrick")

plt.title("Monthly Crime Incidents in Boston (2020–Sep 2025) With Cleaned Dataset", fontsize=20)
plt.xlabel("Month", fontsize=15)  # Increased font size
plt.ylabel("Number of Incidents", fontsize=20) # Increased font size
plt.grid(alpha=0.3)

# Format x-axis to show months
plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))  # e.g., Jan 2024

plt.xticks(rotation=45, ha='right', fontsize=15) # Increased font size for ticks
plt.yticks(fontsize=20) # Increased font size for ticks
plt.tight_layout()
plt.show()
plt.savefig("monthly_incidents.png")

In [ ]:
print(df['YEAR'].min(), df['YEAR'].max())

2020 2025


# Data Transformation

In [ ]:
def parse_dates_multi_utc(col: pd.Series) -> pd.Series:
    s = col.astype(str).str.strip()
    s = (s.str.replace(r"[…]|\.{3}", "", regex=True)
           .str.replace(r"\s+", " ", regex=True))

    # parse with utc=True every time so results are homogeneous tz-aware
    parsed = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce", utc=True)

    for fmt in [
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %I:%M %p",
        "%m/%d/%Y",
        "%m/%d/%Y %H:%M",
        "%m/%d/%Y %H:%M:%S",
    ]:
        mask = parsed.isna()
        if mask.any():
            parsed.loc[mask] = pd.to_datetime(s[mask], format=fmt, errors="coerce", utc=True)

    # final fallback (dateutil)
    mask = parsed.isna()
    if mask.any():
        parsed.loc[mask] = pd.to_datetime(s[mask], errors="coerce", utc=True)

    return parsed  # dtype should be datetime64[ns, UTC]

df = df.copy()
df['OCCURRED_ON_DATE'] = parse_dates_multi_utc(df['OCCURRED_ON_DATE'])

#  standardize all to UTC and strip time zone info
df['OCCURRED_ON_DATE'] = df['OCCURRED_ON_DATE'].dt.tz_convert('UTC').dt.tz_localize(None)

# compute t since global t0 (in days)
t0 = df['OCCURRED_ON_DATE'].min()
df['t'] = (df['OCCURRED_ON_DATE'] - t0).dt.total_seconds() / 86400.0

# build the nested list
times_by_district = (
    df.dropna(subset=['DISTRICT', 't'])
      .sort_values('t')
      .groupby('DISTRICT')['t']
      .apply(list)
      .to_dict()
)
districts = sorted(times_by_district.keys())
nested_list = [times_by_district[d] for d in districts]
print(districts)

['A1', 'A15', 'A7', 'B2', 'B3', 'C11', 'C6', 'D14', 'D4', 'E13', 'E18', 'E5', 'External', 'Outside of']


In [ ]:
#Roxbury B2, Mattapan B3
import json

df = df.copy()
df['OCCURRED_ON_DATE'] = parse_dates_multi_utc(df['OCCURRED_ON_DATE'])

#  standardize all to UTC and strip time zone info
df['OCCURRED_ON_DATE'] = df['OCCURRED_ON_DATE'].dt.tz_convert('UTC').dt.tz_localize(None)

cutoff = pd.Timestamp('2025-04-01')
df_recent = df[df['OCCURRED_ON_DATE'] >= cutoff].copy()

# recompute t0 and derived values using only recent data
t0 = df_recent['OCCURRED_ON_DATE'].min()
df_recent['t'] = (df_recent['OCCURRED_ON_DATE'] - t0).dt.total_seconds() / 86400.0

# rebuild nested list by district
times_by_district = (
    df_recent.dropna(subset=['DISTRICT', 't'])
             .sort_values('t')
             .groupby('DISTRICT')['t']
             .apply(list)
             .to_dict()
)

districts = sorted(times_by_district.keys())
#print(districts)
nested_list = [times_by_district[d] for d in districts]

#print(df_recent['OCCURRED_ON_DATE'].min(), df_recent['OCCURRED_ON_DATE'].max())
print(nested_list[3])
print(nested_list[4])

print(np.max(np.array(nested_list[3])), np.max(np.array(nested_list[4])))

with open("data.jsonl", "w") as f:
    json.dump(nested_list[3], f)
    f.write("\n")
    json.dump(nested_list[4], f)
    f.write("\n")


[0.0, 0.0, 0.0, 0.0, 0.03125, 0.2743055555555556, 0.3472222222222222, 0.40555555555555556, 0.4222222222222222, 0.4222222222222222, 0.5263888888888889, 0.5395833333333333, 0.5819444444444445, 0.6041666666666666, 0.6229166666666667, 0.6388888888888888, 0.65625, 0.6597222222222222, 0.6611111111111111, 0.6784722222222223, 0.6819444444444445, 0.6979166666666666, 0.7916666666666666, 0.8055555555555556, 0.81875, 0.8638888888888889, 0.8798611111111111, 0.9138888888888889, 0.9138888888888889, 1.0, 1.0, 1.2569444444444444, 1.2604166666666667, 1.3083333333333333, 1.4055555555555554, 1.4159722222222222, 1.4805555555555556, 1.5541666666666667, 1.5541666666666667, 1.625, 1.6465277777777778, 1.66875, 1.6715277777777777, 1.6979166666666667, 1.7104166666666667, 1.8243055555555556, 1.9166666666666667, 1.917361111111111, 2.0, 2.0, 2.0, 2.0, 2.0395833333333333, 2.097916666666667, 2.3520833333333333, 2.3541666666666665, 2.4402777777777778, 2.584722222222222, 2.6194444444444445, 2.6354166666666665, 2.716666

In [ ]:
data_rox = nested_list[3]
data_mat = nested_list[4]

In [ ]:
print(data_mat)
np.max(data_mat)

[0.0, 0.0, 0.10416666666666667, 0.11388888888888889, 0.3958333333333333, 0.40625, 0.4152777777777778, 0.5388888888888889, 0.5527777777777778, 0.7277777777777777, 0.7479166666666667, 0.7493055555555556, 0.7722222222222223, 0.7888888888888889, 1.0173611111111112, 1.0625, 1.24375, 1.3125, 1.4826388888888888, 1.5020833333333334, 1.5347222222222223, 1.6041666666666667, 1.6284722222222223, 1.6381944444444445, 1.6979166666666667, 1.7013888888888888, 1.7409722222222221, 1.7791666666666666, 1.8041666666666667, 1.8375, 1.9034722222222222, 2.0, 2.1, 2.25, 2.390277777777778, 2.5083333333333333, 2.6486111111111112, 2.6875, 2.7041666666666666, 2.8541666666666665, 2.8777777777777778, 2.910416666666667, 2.9694444444444446, 3.0, 3.0, 3.310416666666667, 3.3368055555555554, 3.3645833333333335, 3.3680555555555554, 3.4875, 3.4881944444444444, 3.5, 3.6347222222222224, 3.6625, 3.6930555555555555, 3.7125, 3.73125, 3.7930555555555556, 3.879166666666667, 3.9430555555555555, 4.0, 4.0, 4.0625, 4.272222222222222, 

np.float64(182.83194444444445)